# K-Means Clustering — ML Practical Notebook

## Cheat-sheet section to locate
Open **Page 2 → “3. Unsupervised Learning Models (Syntactic)” → “K Means”** in `cheet-10.pdf`.

The source gives these core elements:
- `KMeans(n_clusters=<int>, random_state=<int>)`
- `kmeans.fit(X)`
- `wcss = kmeans.inertia_`
- `labels = kmeans.labels_`
- `centers = kmeans.cluster_centers_`
- `silhouette_score(X, labels)` (higher is generally better; range −1 to 1)

The cheat sheet's minimal example uses `n_clusters=3` and `random_state=42`. It does not prescribe an optimal-k selection method, scaling strategy, or PCA for K-Means; those are added below as practical extensions and are labelled accordingly.

## 1. Imports and load dataset

In [ ]:

import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    silhouette_score, calinski_harabasz_score, davies_bouldin_score
)
from sklearn.decomposition import PCA

csv_files = sorted(glob.glob("*.csv"))
if not csv_files:
    file_path = input("Enter path to your CSV file: ").strip().strip('"')
elif len(csv_files) == 1:
    file_path = csv_files[0]
    print("Using:", file_path)
else:
    for i, name in enumerate(csv_files):
        print(f"{i}: {name}")
    file_path = csv_files[int(input("Choose CSV number: ").strip())]

df = pd.read_csv(file_path)
print("Shape:", df.shape)
display(df.head())
print("Columns:", df.columns.tolist())


## 2. Choose features
K-Means is unsupervised: it does not require a target column. Exclude identifier columns and non-feature text. If your dataset has a known class/target column, do **not** use it as an input feature; you can keep it separately only for later external comparison.

In [ ]:

drop_cols_text = input(
    "Enter columns to exclude, comma-separated (e.g. ID,Target), or press Enter for none: "
).strip()
drop_cols = [c.strip() for c in drop_cols_text.split(",") if c.strip()]
unknown = [c for c in drop_cols if c not in df.columns]
if unknown:
    raise ValueError(f"These columns do not exist: {unknown}")

X_df = df.drop(columns=drop_cols).copy()

# This basic version uses numeric columns only; encode categorical columns separately if required.
X_df = X_df.select_dtypes(include=np.number)
if X_df.shape[1] == 0:
    raise ValueError("No numeric feature columns remain. Encode categorical features before K-Means.")

print("Features used:", X_df.columns.tolist())
print("Numeric feature matrix shape:", X_df.shape)
display(X_df.describe().T)
print("\nMissing values:")
display(X_df.isna().sum().to_frame("missing_count"))


## 3. Handle missing values and scale features
**Practical extension:** K-Means uses distances, so features with large numeric scales can dominate. Standardizing numeric features is commonly useful. The core cheat sheet only specifies KMeans and does not explicitly require scaling for K-Means.

In [ ]:

imputer = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(X_df)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

print("Prepared matrix shape:", X_scaled.shape)


## 4. Fit K-Means with the cheat-sheet syntax

In [ ]:

# Cheat-sheet minimal example: n_clusters=3, random_state=42
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans.fit(X_scaled)

wcss = kmeans.inertia_
labels = kmeans.labels_
centers = kmeans.cluster_centers_

print("WCSS / inertia:", wcss)
print("Cluster labels (first 20):", labels[:20])
print("Cluster centers in standardized feature space:")
display(pd.DataFrame(centers, columns=X_df.columns))
print("\nCluster sizes:")
display(pd.Series(labels).value_counts().sort_index().rename("count").to_frame())


## 5. Evaluate clustering quality
The cheat sheet explicitly includes `silhouette_score`. Calinski–Harabasz and Davies–Bouldin are optional additional metrics. Silhouette and Calinski–Harabasz are generally better when higher; Davies–Bouldin is generally better when lower.

In [ ]:

if len(np.unique(labels)) > 1 and len(np.unique(labels)) < len(labels):
    sil = silhouette_score(X_scaled, labels)
    ch = calinski_harabasz_score(X_scaled, labels)
    db = davies_bouldin_score(X_scaled, labels)
    print("Silhouette score:", round(sil, 4))
    print("Calinski-Harabasz score:", round(ch, 4))
    print("Davies-Bouldin score:", round(db, 4))
else:
    print("Cluster metrics cannot be calculated: clustering returned one cluster or one cluster per row.")


## 6. Find a suitable number of clusters (Elbow method)
**Practical extension:** the cheat sheet's minimal example sets `n_clusters=3`; it does not specify the Elbow method. This section tests a range of `k` values and plots WCSS/inertia.

In [ ]:

max_k = min(10, len(X_scaled) - 1)
if max_k < 2:
    raise ValueError("At least 3 rows are needed to compare multiple cluster counts.")

k_values = list(range(2, max_k + 1))
inertias = []
silhouette_values = []

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = model.fit_predict(X_scaled)
    inertias.append(model.inertia_)
    if len(np.unique(cluster_labels)) > 1:
        silhouette_values.append(silhouette_score(X_scaled, cluster_labels))
    else:
        silhouette_values.append(np.nan)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(k_values, inertias, marker="o")
axes[0].set_title("Elbow method")
axes[0].set_xlabel("Number of clusters (k)")
axes[0].set_ylabel("WCSS / inertia")

axes[1].plot(k_values, silhouette_values, marker="o")
axes[1].set_title("Silhouette score by k")
axes[1].set_xlabel("Number of clusters (k)")
axes[1].set_ylabel("Silhouette score")
plt.tight_layout()
plt.show()

best_silhouette_k = k_values[int(np.nanargmax(silhouette_values))]
print("k with highest silhouette score in tested range:", best_silhouette_k)
print("Use the plot and domain context too; do not select k mechanically from one metric alone.")


## 7. Refit with your chosen k and save cluster assignments

In [ ]:

# Change this to the k you decide to use after examining the plots.
chosen_k = best_silhouette_k

final_kmeans = KMeans(n_clusters=chosen_k, random_state=42, n_init=10)
final_labels = final_kmeans.fit_predict(X_scaled)

results = df.copy()
results["cluster"] = final_labels
display(results.head(10))
results.to_csv("kmeans_clustered_data.csv", index=False)
print("Saved kmeans_clustered_data.csv")


## 8. Visualize clusters using PCA (2D)
**Practical extension:** PCA is used here only to project the scaled data to two dimensions for plotting. The clustering itself is fitted on all scaled features, not on the 2D projection.

In [ ]:

if X_scaled.shape[1] >= 2:
    pca = PCA(n_components=2, random_state=42)
    X_2d = pca.fit_transform(X_scaled)

    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=final_labels, s=35)
    plt.xlabel("Principal component 1")
    plt.ylabel("Principal component 2")
    plt.title("K-Means clusters projected into 2D using PCA")
    plt.colorbar(scatter, label="Cluster")
    plt.tight_layout()
    plt.show()

    print("PCA explained variance ratio:", pca.explained_variance_ratio_)
    print("Total variance shown in 2D:", round(float(pca.explained_variance_ratio_.sum()), 4))
else:
    print("At least two numeric features are required for a 2D PCA plot.")


## 9. Interpret cluster profiles
Cluster means are calculated on the original numeric feature scale, which is usually easier to interpret than standardized centroids.

In [ ]:

profile_input = X_df.copy()
profile_input["cluster"] = final_labels
cluster_profile = profile_input.groupby("cluster").mean(numeric_only=True)
display(cluster_profile)


## Quick viva notes
- **Unsupervised learning:** the model finds structure without using a labelled target.
- **`n_clusters`:** number of clusters requested.
- **`random_state=42`:** makes initialization reproducible.
- **`inertia_` / WCSS:** sum of squared distances from points to their assigned cluster centers; lower values tend to occur as k increases, so use the Elbow plot rather than minimizing WCSS alone.
- **`labels_`:** cluster assignment for each row used in fitting.
- **`cluster_centers_`:** centroid coordinates in the feature space used for fitting.
- **Silhouette score:** ranges from −1 to 1; higher generally indicates better-separated clusters.
- **Why scale?** K-Means uses Euclidean distances, so features on large scales can dominate.
- **Limitations:** K-Means is sensitive to outliers and initialization, and works best for roughly compact/spherical clusters.